# 03 — Equivalent entity/relation lookup through LaminDB

LaminDB catalogs exact-ID registries and generic edge/evidence records in `jkobject/jouvencekb`. It is a query/catalog surface, not a replacement for canonical Parquet. Current ingestion is partial, so notebook output must not hide coverage gaps.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "public":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = REPO_ROOT / "artifacts" / "cache" / "public-notebooks"
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})


In [ ]:
from manage_db.public_notebooks import (
    LAMIN_INSTANCE,
    query_lamindb_edges,
    query_lamindb_evidence,
    query_lamindb_node,
)

print("Exact allowed instance:", LAMIN_INSTANCE)
LIVE_LAMIN = os.environ.get("JOUVENCE_LAMIN_LIVE", "0") == "1"
if LIVE_LAMIN:
    lamin_node = query_lamindb_node("gene", "ENSG00000141510", limit=5)
    lamin_rows = query_lamindb_edges(
        relation="disease_associated_gene",
        x_id="ENSG00000141510",
        limit=20,
    )
    lamin_evidence = query_lamindb_evidence(
        relation="disease_associated_gene",
        x_id="ENSG00000141510",
        limit=20,
    )
    display(lamin_node)
    display(lamin_rows)
    display(lamin_evidence)
else:
    print("Live Lamin query skipped. Set JOUVENCE_LAMIN_LIVE=1 only in an authenticated environment.")


In [ ]:
from manage_db.public_notebooks import diseases_with_gene_evidence

if MODE == "fixture":
    parquet_equivalent = diseases_with_gene_evidence(KG_ROOT, "ENSG00000141510", limit=20)
    display(parquet_equivalent)
print("An empty Lamin result can mean 'not ingested yet'; compare with canonical Parquet before concluding absence.")


Lamin queries are bounded and read-only. The helper refuses any instance slug other than `jkobject/jouvencekb`. This protects against silently querying a similarly named or stale instance and preserves the current ingestion-incompleteness caveat.
